# AIaaS — Cross-Framework Inference Benchmark (optimum-benchmark)

The *cross-framework* tier: run **the same model** through **PyTorch**, **ONNX
Runtime (CUDA EP)**, and **ONNX Runtime (TensorRT EP)** with one standardized
harness — HuggingFace **`optimum-benchmark`** — so the only variable is the
inference backend. It answers "how much does an optimized runtime buy me on this
GPU?" without hand-rolled timing.

Reports per backend:
- **decode throughput** (tokens/s) and **prefill latency**
- **per-token latency** and **peak VRAM**

### How it stays comparable
- One harness (`optimum-benchmark`) drives every backend with identical input
  shapes, warmup, and generation settings — no bespoke timing code.
- Same model + same `(batch_size, sequence_length, max_new_tokens)` across
  backends; only the runtime changes.

### Requirements
- A **CUDA GPU** runtime. The TensorRT EP needs TensorRT libs present (often only
  on A100-class images); it's optional and degrades gracefully if unavailable.
- Model defaults to **`gpt2`** because it exports to ONNX cleanly everywhere.
  Swap `MODEL` for a small ungated LLM (e.g. `Qwen/Qwen2.5-0.5B-Instruct`) once
  you've confirmed it exports on your platform.

> Runs belong in a fork session scoped to `optimum` / `optimum-benchmark` (see
> the strategy doc). This notebook is portable and self-contained so it can be
> dropped into that session and run top to bottom.

## 1. Install optimum-benchmark (+ ONNX Runtime GPU)

In [ ]:
import subprocess, sys

# onnxruntime-gpu provides the CUDA and TensorRT execution providers.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "optimum-benchmark[onnxruntime-gpu]", "pandas"], check=False)

try:
    import optimum_benchmark, onnxruntime
    print("optimum-benchmark", getattr(optimum_benchmark, "__version__", "?"),
          "| onnxruntime", getattr(onnxruntime, "__version__", "?"))
    print("ORT providers:", onnxruntime.get_available_providers())
except Exception as e:
    print("Imports not ready:", e)
    print("If a CUDA lib was upgraded, RESTART the runtime and re-run from this cell.")


## 2. Environment & hardware capture
Tagged onto every result so platforms stay comparable.

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
One model, identical shapes across backends. The TensorRT EP is only attempted if
ONNX Runtime exposes it on this platform.

In [ ]:
import onnxruntime

MODEL = "gpt2"             # exports to ONNX cleanly; swap for a small ungated LLM
TASK = "text-generation"
# Precision is held CONSTANT across backends so the *only* variable is the runtime.
# ONNX Runtime exports/runs fp32 by default, so PyTorch is set to fp32 too for an
# apples-to-apples comparison. (fp16 is possible, but you must ALSO enable fp16 on
# the ORT side or it becomes PyTorch-fp16 vs ORT-fp32 — see the precision warning.)
DTYPE = "float32"

BATCH_SIZE = 1
SEQ_LEN = 128             # prompt length
GEN_TOKENS = 128         # tokens to generate (fixed for comparability)
WARMUP_RUNS = 10

# Always benchmark PyTorch + ONNX Runtime (CUDA). Add the TensorRT EP only if the
# installed onnxruntime actually exposes it on this platform.
BACKENDS = ["pytorch", "onnxruntime-cuda"]
if "TensorrtExecutionProvider" in onnxruntime.get_available_providers():
    BACKENDS.append("onnxruntime-tensorrt")

OUT_DIR = "optimum_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL={MODEL}, TASK={TASK}, backends={BACKENDS}")


## 4. Run each backend through the same harness
Each backend runs in its own subprocess (clean GPU state). A failing backend (for
example a TensorRT export that won't build) is recorded as an error and skipped,
so the others still produce numbers.

In [ ]:
from optimum_benchmark import (
    Benchmark, BenchmarkConfig, ProcessConfig,
    InferenceConfig, PyTorchConfig, ONNXRuntimeConfig,
)
from optimum_benchmark.logging_utils import setup_logging

setup_logging(level="WARNING")

launcher = ProcessConfig(device_isolation=False)
scenario = InferenceConfig(
    latency=True, memory=True, warmup_runs=WARMUP_RUNS,
    input_shapes={"batch_size": BATCH_SIZE, "sequence_length": SEQ_LEN},
    generate_kwargs={"max_new_tokens": GEN_TOKENS, "min_new_tokens": GEN_TOKENS},
)

def make_backend(kind):
    # PyTorch runs at DTYPE; ORT runs at its export default (fp32). Keep DTYPE
    # = float32 for a precision-matched comparison.
    common = dict(model=MODEL, device="cuda", device_ids="0", task=TASK)
    if kind == "pytorch":
        return PyTorchConfig(torch_dtype=DTYPE, **common)
    if kind == "onnxruntime-cuda":
        return ONNXRuntimeConfig(provider="CUDAExecutionProvider", **common)
    if kind == "onnxruntime-tensorrt":
        return ONNXRuntimeConfig(provider="TensorrtExecutionProvider", **common)
    raise ValueError(f"unknown backend {kind}")

REPORTS = {}
for kind in BACKENDS:
    print(f"\n=== {kind} ===")
    try:
        cfg = BenchmarkConfig(name=kind, launcher=launcher,
                              scenario=scenario, backend=make_backend(kind))
        report = Benchmark.launch(cfg)
        REPORTS[kind] = report.to_dict()
        print("done")
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        REPORTS[kind] = {"error": f"{type(e).__name__}: {e}"}


## 5. Results — normalize, save, summarize
optimum-benchmark's report schema shifts a little between versions, so headline
metrics are plucked defensively; the full per-backend report is always saved.

In [ ]:
import pandas as pd

def pluck(d, *path, default=None):
    cur = d
    for k in path:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

def summarize(rep):
    if not isinstance(rep, dict) or "error" in rep:
        return {"status": rep.get("error", "no report")[:40] if isinstance(rep, dict) else "no report"}
    # decode throughput (tokens/s) and latencies; keys vary across versions.
    decode_tps = pluck(rep, "decode", "throughput", "value")
    prefill_lat = pluck(rep, "prefill", "latency", "mean")
    per_tok = pluck(rep, "per_token", "latency", "mean") or pluck(rep, "decode", "latency", "mean")
    vram = (pluck(rep, "decode", "memory", "max_allocated")
            or pluck(rep, "decode", "memory", "max_vram")
            or pluck(rep, "decode", "memory", "max_global_vram"))
    return {
        "decode throughput": round(decode_tps, 2) if isinstance(decode_tps, (int, float)) else "-",
        "prefill latency": round(prefill_lat, 5) if isinstance(prefill_lat, (int, float)) else "-",
        "per-token latency": round(per_tok, 6) if isinstance(per_tok, (int, float)) else "-",
        "max VRAM": round(vram, 1) if isinstance(vram, (int, float)) else "-",
    }

def precision_of(kind):
    return DTYPE if kind == "pytorch" else "fp32 (ORT export default)"

HEADLINE = ("decode throughput", "prefill latency", "per-token latency", "max VRAM")
SUMMARY = {}
for k, v in REPORTS.items():
    row = summarize(v)
    if "status" not in row:
        row = {"precision": precision_of(k), **row}
        if all(row[m] == "-" for m in HEADLINE):
            print(f"WARNING: {k} ran but no headline metric matched the known report\n"
                  f"         paths (extraction issue, not a failed benchmark). Report keys: "
                  f"{list(v.keys())}")
    SUMMARY[k] = row

if DTYPE != "float32":
    print(f"\nWARNING: DTYPE={DTYPE} but ONNX Runtime runs fp32 by default \u2014 this is "
          "NOT a precision-matched comparison (PyTorch-" + DTYPE + " vs ORT-fp32).")

NORM = {
    "schema": "optimum-bench/1.0",
    "env": ENV, "model": MODEL, "task": TASK, "dtype": DTYPE,
    "input_shapes": {"batch_size": BATCH_SIZE, "sequence_length": SEQ_LEN},
    "generate_tokens": GEN_TOKENS,
    "precision_matched": DTYPE == "float32",
    "summary": SUMMARY,
    "reports": REPORTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"optimum_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)

print("Saved:", out)
print("\n==== CROSS-FRAMEWORK SUMMARY ====")
print(f"{ENV['gpu_name']} | {MODEL} | bs={BATCH_SIZE} seq={SEQ_LEN} gen={GEN_TOKENS}")
print("(units as reported by optimum-benchmark; see saved JSON for exact units)\n")
df = pd.DataFrame(SUMMARY).T
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string())


## 6. Reading the result (and what's next)

- **decode throughput** is the headline: how many tokens/s each runtime sustains
  on this GPU for the same model. ONNX Runtime (and especially the TensorRT EP)
  typically beats eager PyTorch; the gap is the "optimized runtime" win.
- A backend showing `status` instead of numbers failed to build/export on this
  platform (common for the TensorRT EP) — the rest are still valid.

**Where this fits:** this is the *cross-framework* tier. For production serving
numbers use `vllm_serving_benchmark.ipynb`; for the peak A100 ceiling, TensorRT-LLM
is the next rung. See `BENCHMARKING_STRATEGY.md`. Per the tooling plan, the actual
runs live in a fork session scoped to `optimum` / `optimum-benchmark`.